# AI Model Validation: From Mathematics to Trust
## Episode 10 — Robustness and Stress Testing: What Happens When the World Changes?

**Author: Vishwanath Akuthota**  
**Founder of DrPinnacle, OpenVals and The Foundry's**

A model can perform extremely well on its test set and still fail when reality changes.

> **Does the model continue to work when the world is no longer identical to the evaluation dataset?**

This episode develops robustness scientifically through perturbations, noise, missingness, distribution shift, tail risk, adversarial sensitivity, subgroup stress testing, degradation curves, uncertainty and continuous validation.

> **Average performance tells us how a model behaved. Robustness tells us how easily that behaviour can break.**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import platform, sys

print("Python :", sys.version.split()[0])
print("System :", platform.system(), platform.release())
print("Machine:", platform.machine())
print("NumPy  :", np.__version__)


## 1. From Generalization to Robustness

Evaluation risk under distribution P:

### R_P(f) = E_P[L(Y, f(X))]

Deployment may follow another distribution Q:

### R_Q(f) = E_Q[L(Y, f(X))]

Therefore:

### Good R_P(f) does not imply good R_Q(f)

Generalization asks whether evidence transfers beyond training data. Robustness asks how stable that performance remains when operating conditions are deliberately or naturally changed.


## 2. High Performance Is Not High Robustness

A clean metric describes performance under a specified evaluation condition.

Robustness describes performance stability under change.

### High Performance ≠ High Robustness

A scientifically useful robustness claim must identify the stressor, magnitude, population, metric and acceptance criterion.


## 3. Controlled Regression Laboratory

Consider:

### Y = 2X + 1 + ε

where ε is random noise.

Evaluate the slightly imperfect model:

### Ŷ = 1.95X + 1.05

This simple system lets us observe degradation without hiding the mathematics inside a complex model.


In [ ]:
rng=np.random.default_rng(1010)
n=2000
x=rng.normal(0,1,n)
y=2*x+1+rng.normal(0,.5,n)

def model(x):
    return 1.95*x+1.05

def mae(y,p):
    return np.mean(np.abs(y-p))

def rmse(y,p):
    return np.sqrt(np.mean((y-p)**2))

baseline_mae=mae(y,model(x))
baseline_rmse=rmse(y,model(x))
print("Baseline MAE :", baseline_mae)
print("Baseline RMSE:", baseline_rmse)


## 4. Perturbation Testing

For input perturbation δ:

### X' = X + δ

Prediction sensitivity can be measured as:

### Δf = |f(X + δ) − f(X)|

For a linear model f(x)=wx+b:

### f(x+δ) − f(x) = wδ

therefore:

### |Δf| = |w||δ|

The model's slope directly controls sensitivity in this one-dimensional example.


In [ ]:
deltas=np.linspace(0,1.5,100)
change=[np.mean(np.abs(model(x+d)-model(x))) for d in deltas]

plt.figure(figsize=(8,5))
plt.plot(deltas,change)
plt.xlabel("Perturbation magnitude |δ|")
plt.ylabel("Mean absolute prediction change")
plt.title("Prediction Sensitivity to Perturbation")
plt.grid(alpha=.25)
plt.show()


## 5. Lipschitz-Style Stability

A function is K-Lipschitz when:

### ||f(x₁) − f(x₂)|| ≤ K ||x₁ − x₂||

K bounds how rapidly outputs can change relative to inputs under the selected norms.

A smaller K may indicate greater local stability, but Lipschitz behaviour alone does not establish complete real-world robustness.


## 6. Noise Stress Testing

Suppose measurements are corrupted:

### X_noise = X + ε_noise

with:

### ε_noise ~ N(0, σ²)

We now measure model error as σ increases.


In [ ]:
noise_levels=np.linspace(0,2,21)
maes=[]
rmses=[]

for sigma in noise_levels:
    noisy=x+rng.normal(0,sigma,len(x))
    maes.append(mae(y,model(noisy)))
    rmses.append(rmse(y,model(noisy)))

plt.figure(figsize=(8,5))
plt.plot(noise_levels,maes,marker="o",label="MAE")
plt.plot(noise_levels,rmses,marker="o",label="RMSE")
plt.xlabel("Noise standard deviation σ")
plt.ylabel("Error")
plt.title("Performance Degradation Under Measurement Noise")
plt.legend()
plt.grid(alpha=.25)
plt.show()


## 7. Degradation Curves

For stress severity s and error E(s):

### D(s) = E(s) − E(0)

Relative degradation:

### D_rel(s) = [E(s) − E(0)] / E(0)

A robustness curve reveals not only whether performance worsens, but **how quickly it worsens**.


In [ ]:
relative=(np.array(rmses)-rmses[0])/rmses[0]

plt.figure(figsize=(8,5))
plt.plot(noise_levels,relative,marker="o")
plt.axhline(0,linestyle="--")
plt.xlabel("Stress severity σ")
plt.ylabel("Relative RMSE degradation")
plt.title("Relative Robustness Degradation")
plt.grid(alpha=.25)
plt.show()


## 8. Robustness Boundary

Suppose maximum acceptable error is E_max.

Define:

### s* = sup { s : E(s) ≤ E_max }

s* is an operational robustness boundary: the largest stress severity for which the model remains inside the stated requirement.


In [ ]:
E_max=1.5
acceptable=np.array(rmses)<=E_max
boundary=noise_levels[np.where(acceptable)[0][-1]] if acceptable.any() else np.nan
print("Maximum acceptable RMSE:",E_max)
print("Largest tested acceptable severity:",boundary)


## 9. Missing-Data Stress Testing

Production information may be incomplete.

Let m denote the missing-data rate:

### M(m) = Performance at missingness m

The missingness mechanism matters:

### MCAR — Missing Completely at Random
### MAR — Missing at Random conditional on observed information
### MNAR — Missing Not at Random

Robustness to MCAR does not imply robustness to systematic missingness.


In [ ]:
rng2=np.random.default_rng(1011)
X=rng2.normal(0,1,(2000,3))
weights=np.array([2,-1.2,.7])
target=X@weights+rng2.normal(0,.5,2000)

rates=np.linspace(0,.8,9)
missing_rmse=[]

for rate in rates:
    Xm=X.copy()
    mask=rng2.random(Xm.shape)<rate
    Xm[mask]=0  # controlled demonstration only
    missing_rmse.append(rmse(target,Xm@weights))

plt.figure(figsize=(8,5))
plt.plot(rates,missing_rmse,marker="o")
plt.xlabel("Missing-value fraction")
plt.ylabel("RMSE")
plt.title("Stress Test: Missing Information")
plt.grid(alpha=.25)
plt.show()


The fixed-value replacement above is intentionally simple. In real validation, the imputation strategy is itself part of the evaluated system.


## 10. Covariate Shift

Covariate shift occurs when:

### P_train(X) ≠ P_deploy(X)

while approximately:

### P_train(Y|X) ≈ P_deploy(Y|X)

The relationship may remain valid, but deployment moves into different regions of input space.


In [ ]:
shifts=np.linspace(0,4,17)
cov_rmse=[]

for mu in shifts:
    xd=rng.normal(mu,1,2000)
    yd=2*xd+1+rng.normal(0,.5,len(xd))
    cov_rmse.append(rmse(yd,model(xd)))

plt.figure(figsize=(8,5))
plt.plot(shifts,cov_rmse,marker="o")
plt.xlabel("Deployment input mean shift")
plt.ylabel("RMSE")
plt.title("Performance Under Covariate Shift")
plt.grid(alpha=.25)
plt.show()


## 11. Concept Shift

Concept shift changes the conditional relationship:

### P_train(Y|X) ≠ P_deploy(Y|X)

For example, training may follow:

### Y = 2X + 1 + ε

while deployment evolves toward:

### Y = aX + 1 + ε

Covariate shift and concept shift are different failure mechanisms and should not be collapsed into a vague notion of “drift.”


In [ ]:
slopes=np.linspace(2,4,17)
concept_rmse=[]

for a in slopes:
    xd=rng.normal(0,1,2000)
    yd=a*xd+1+rng.normal(0,.5,len(xd))
    concept_rmse.append(rmse(yd,model(xd)))

plt.figure(figsize=(8,5))
plt.plot(slopes,concept_rmse,marker="o")
plt.xlabel("True deployment slope")
plt.ylabel("RMSE")
plt.title("Performance Under Concept Shift")
plt.grid(alpha=.25)
plt.show()


## 12. Average Risk and Worst-Case Risk

Average risk:

### R_avg = E[L]

Worst-case observed loss:

### R_worst = max L

For a family of plausible deployment distributions 𝒬:

### R_robust(f) = sup_(Q∈𝒬) R_Q(f)

The uncertainty set 𝒬 is itself a scientific assumption. A worst-case result is meaningful only relative to the set over which the worst case was defined.


In [ ]:
residual=np.abs(y-model(x))
print("Mean absolute error:",residual.mean())
print("95th percentile    :",np.quantile(residual,.95))
print("99th percentile    :",np.quantile(residual,.99))
print("Maximum error      :",residual.max())


## 13. Tail Robustness

For absolute error |e|, define the q-th quantile:

### Q_q = inf { t : P(|e| ≤ t) ≥ q }

Q₀.₉₅ and Q₀.₉₉ expose severe errors that mean performance can hide.

### Low Average Error ≠ Low Tail Risk


## 14. Subgroup Stress Testing

For subgroup g:

### R_g(f) = E[L(Y,f(X)) | G=g]

Aggregate results can hide a fragile subgroup.

### Aggregate Robustness ≠ Subgroup Robustness


In [ ]:
rng3=np.random.default_rng(1012)
group=np.array(["A"]*1500+["B"]*500)
xg=rng3.normal(0,1,2000)
yg=np.empty(2000)

A=group=="A"
B=group=="B"
yg[A]=2*xg[A]+1+rng3.normal(0,.5,A.sum())
yg[B]=3*xg[B]+1+rng3.normal(0,.8,B.sum())

for g in ["A","B"]:
    mask=group==g
    print(g,"N=",mask.sum(),"RMSE=",rmse(yg[mask],model(xg[mask])))
print("Aggregate RMSE:",rmse(yg,model(xg)))


## 15. Perturbation Sets and Adversarial Robustness

Define allowed perturbations:

### Δ = { δ : ||δ|| ≤ ε }

Worst-case local loss:

### max_(δ∈Δ) L(y, f(x+δ))

An adversarial perturbation is:

### δ* = argmax_(||δ||≤ε) L(y,f(x+δ))

Random noise and adversarial perturbations answer different questions.

### Random Disturbance ≠ Strategically Selected Disturbance


## 16. Gradient Sensitivity

For differentiable f:

### f(x+δ) ≈ f(x) + ∇f(x)ᵀδ

Under an L2 constraint:

### ||δ||₂ ≤ ε

the first-order maximum output change is approximately bounded by:

### ε ||∇f(x)||₂

Large gradients identify locally sensitive regions.


In [ ]:
def nonlinear_model(x):
    return np.sin(2*x)+.2*x**2

points=np.linspace(-3,3,300)
h=1e-4
gradient=(nonlinear_model(points+h)-nonlinear_model(points-h))/(2*h)

plt.figure(figsize=(8,5))
plt.plot(points,np.abs(gradient))
plt.xlabel("x")
plt.ylabel("|df/dx|")
plt.title("Local Sensitivity Across Input Space")
plt.grid(alpha=.25)
plt.show()


## 17. Invariance

If transformation T should preserve task meaning:

### f(T(x)) ≈ f(x)

Robustness testing must distinguish transformations that **should** preserve the answer from transformations that legitimately should change it.

Blind invariance is not robustness.


## 18. Metamorphic Testing

Sometimes the exact correct output is unavailable, but relationships between outputs are known.

A metamorphic relation can require:

### f(T(x)) = T'(f(x))

or approximately:

### d[f(T(x)), T'(f(x))] ≤ ε

This allows systematic validation even when a conventional test oracle is incomplete.


## 19. Stress-Test Matrix

A useful experimental structure is:

### Scenario = Stressor × Severity × Subgroup × Metric

Examples:

- noise × low/medium/high × subgroup × RMSE
- missingness × 10/30/50% × segment × recall
- shift × mild/moderate/severe × geography × calibration
- perturbation × ε × model version × failure rate

Robustness is better represented as a profile than as a single number.


## 20. Best Clean Model ≠ Most Robust Model


In [ ]:
stress=np.linspace(0,2,21)
model_A=.45+.70*stress**1.5
model_B=.60+.28*stress

plt.figure(figsize=(8,5))
plt.plot(stress,model_A,label="Model A: better clean score")
plt.plot(stress,model_B,label="Model B: slower degradation")
plt.xlabel("Stress severity")
plt.ylabel("Error")
plt.title("Clean Performance and Robustness Can Rank Models Differently")
plt.legend()
plt.grid(alpha=.25)
plt.show()


A model that wins on clean data can lose under realistic stress. Model selection should therefore reflect intended operating conditions.


## 21. Robustness Uncertainty

Stress-test results are estimates.

For stress losses L₁,...,L_n:

### L̄ = (1/n) ΣLᵢ

Under independent sampling:

### SE(L̄) = s_L / √n

A degradation number without uncertainty can create false precision.


In [ ]:
stress_x=x+rng.normal(0,1,len(x))
stress_loss=(y-model(stress_x))**2
mean_loss=stress_loss.mean()
se=stress_loss.std(ddof=1)/np.sqrt(len(stress_loss))

print("Mean stressed MSE:",mean_loss)
print("Approximate SE    :",se)
print("Approximate 95% interval:",
      (mean_loss-1.96*se,mean_loss+1.96*se))


## 22. Bootstrap the Degradation

Define:

### Δ = RMSE_stress − RMSE_baseline

Bootstrap resampling estimates the sampling variability of measured degradation.


In [ ]:
rng4=np.random.default_rng(1013)
sx=x+rng4.normal(0,1,len(x))
base=model(x)
stressed=model(sx)

B=2000
delta=np.empty(B)

for b in range(B):
    idx=rng4.integers(0,len(x),len(x))
    e0=np.sqrt(np.mean((y[idx]-base[idx])**2))
    e1=np.sqrt(np.mean((y[idx]-stressed[idx])**2))
    delta[b]=e1-e0

print("Mean degradation:",delta.mean())
print("95% percentile interval:",np.quantile(delta,[.025,.975]))

plt.figure(figsize=(8,5))
plt.hist(delta,bins=40)
plt.xlabel("RMSE degradation")
plt.ylabel("Frequency")
plt.title("Uncertainty in Robustness Degradation")
plt.grid(alpha=.2)
plt.show()


## 23. Failure Probability

Define a failure event:

### F = 1[L > L_max]

Then:

### p_fail = P(L > L_max)

Observed failure rate:

### p̂_fail = (1/n) ΣFᵢ

For high-consequence systems, failure probability can be more operationally meaningful than average error.


In [ ]:
error_limit=2.0
absolute_error=np.abs(y-stressed)
failure=absolute_error>error_limit

print("Error limit:",error_limit)
print("Observed failure rate:",failure.mean())
print("Failures:",failure.sum(),"of",len(failure))


## 24. Robustness Budget

An engineering requirement can specify:

### D(s) ≤ D_max

or:

### p_fail(s) ≤ p_max

A meaningful robustness claim therefore answers:

### Robust to WHAT?
### Over WHAT magnitude?
### For WHICH population?
### Using WHICH metric?
### Under WHICH acceptance criterion?


## 25. Plausible Stress Matters

An unrealistically extreme perturbation may demonstrate mathematical fragility without representing deployment risk.

Testing only convenient mild perturbations can create false confidence.

Stress scenarios should be grounded in expected operational variation, historical incidents, data-quality failures, domain constraints, adversarial threat models, and relevant safety or governance requirements.


## 26. Robustness, Reliability and Safety

Reliability concerns consistency of intended performance.

Robustness concerns stability when conditions change.

Safety concerns unacceptable harm.

### Robustness ≠ Reliability ≠ Safety

These dimensions interact, but none substitutes for the others.


## 27. Continuous Robustness Validation

After deployment, monitor:

### Performance(t)
### Input Distribution(t)
### Calibration(t)
### Failure Rate(t)
### Robustness Indicators(t)

A pre-deployment robustness result is not a permanent certificate. Material environmental change can invalidate earlier evidence.


## 28. OpenVals Perspective

A serious validation platform should preserve:

- baseline metrics
- stressor definition
- severity
- perturbation model
- corruption type
- subgroup
- distribution-shift scenario
- degradation curve
- robustness boundary
- tail performance
- failure probability
- uncertainty intervals
- model-version comparison
- acceptance criterion
- temporal drift
- post-deployment incidents

The objective is not to label a model **robust**.

The objective is to make robustness claims **measurable, reproducible, challengeable and continuously testable**.


## 29. Scientific Robustness Chain

### Baseline → Stressor → Severity → Performance Degradation → Uncertainty → Failure Boundary → Evidence → Operational Decision

This transforms robustness from an adjective into an experimental object.


## 30. Mathematical Summary

### R_P(f) = E_P[L(Y,f(X))]

### R_Q(f) = E_Q[L(Y,f(X))]

### X' = X + δ

### Δf = |f(X+δ) − f(X)|

### ||f(x₁)−f(x₂)|| ≤ K||x₁−x₂||

### D(s) = E(s) − E(0)

### D_rel(s) = [E(s)−E(0)] / E(0)

### s* = sup{s : E(s) ≤ E_max}

### R_robust(f) = sup_(Q∈𝒬) R_Q(f)

### δ* = argmax_(||δ||≤ε) L(y,f(x+δ))

### f(x+δ) ≈ f(x) + ∇f(x)ᵀδ

### p_fail = P(L > L_max)

### SE(L̄) = s_L / √n


## 31. Central Conclusion

### Clean Performance ≠ Robust Performance

> **A model is not robust because it succeeds under ideal conditions. Robustness is evidence about how gracefully it fails when those conditions change.**

Noise exposes sensitivity.

Missingness exposes dependency.

Covariate shift moves the model into different input regions.

Concept shift can invalidate learned relationships.

Adversarial perturbation probes targeted fragility.

Subgroup stress testing exposes failures hidden by averages.

The scientific goal is not to prove that a model can never fail.

It is to characterize:

### Where does it fail?
### How quickly does performance degrade?
### Under which conditions?
### With what uncertainty?
### At what point should we stop trusting the system?

## Next Episode

**Episode 11 — Distribution Shift and Model Drift: When Yesterday's Evidence Stops Being Valid**

Episode 11 will formalize covariate shift, label shift, concept drift, temporal drift, PSI, KL divergence, Jensen–Shannon divergence, Wasserstein distance, drift detection, performance decay and continuous-validation triggers.
